# 01 — FP vs ArGEnT vs PointNetMLPJoint

This notebook performs the paper-facing **validation-split evaluation** for the main edge-representation comparison. It reconstructs checkpoints dynamically from the repository, validates local HDF5 assets, runs inference on a deterministic geometry-level holdout, and saves comparison artifacts under `Comparison/results/01_fp_vs_argent`.

**Scope**
- Regimes: `Uniform + Edge`, `Zonal + Edge`
- Families: `ArGEnT_self_att_noSDF`, `PointNetMLPJoint`, `PointNetMLPJoint_FP`
- Shared evaluation split: seed `42`, fraction `0.20`

In [ ]:
from __future__ import annotations
import ast, hashlib, importlib.util, json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
try:
    import torch
    import h5py
except ImportError as exc:
    raise RuntimeError('Install torch and h5py in the selected notebook kernel before executing this comparison.') from exc

_clean_kernel_guard = {'checkpoint_report', 'selected_checkpoints', 'all_samples', 'split_records', 'node_results', 'pooled_metrics', 'geometry_metrics'}
_preexisting = sorted(name for name in _clean_kernel_guard if name in globals())
if _preexisting:
    raise RuntimeError(f'Run this notebook from a clean kernel; found pre-existing globals: {_preexisting}')

CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
if not (REPO_ROOT / 'Uniform').exists():
    raise RuntimeError(f'Repository root not found from {CURRENT_DIR}')
COMPARISON_DIR = REPO_ROOT / 'Comparison'
RESULTS_DIR = COMPARISON_DIR / 'results' / '01_fp_vs_argent'
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(COMPARISON_DIR))
import eval_helpers as eh

SPLIT_SEED, EVAL_FRACTION = 42, 0.20
EVALUATION_LABEL = 'validation-split evaluation'
FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint', 'PointNetMLPJoint_FP']
DATASETS = {
    'Uniform': REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_uniform.h5',
    'Zonal': REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_zonal.h5',
}
COMMIT = __import__('subprocess').check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
VERSIONS = {
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'matplotlib': plt.matplotlib.__version__,
    'torch': getattr(torch, '__version__', 'unknown'),
    'h5py': getattr(h5py, '__version__', 'unknown'),
}

display(Markdown(
    f'**Commit:** `{COMMIT}`\n\n'
    f'**Results directory:** `{RESULTS_DIR}`\n\n'
    f'**Evaluation label:** **{EVALUATION_LABEL}**'
))

## Checkpoint discovery and compatibility audit

This cell discovers checkpoints dynamically under `Uniform/Edge/*/Trained_models` and `Zonal/Edge/*/Trained_models`, validates required architecture/normalization fields, and selects at most one checkpoint per `(regime, family)` without silently substituting incompatible candidates.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def decode(value):
    return value.decode() if isinstance(value, bytes) else value


def _required_checkpoint_status(payload):
    required = ['arch', 'model_state', 'target_mean', 'target_std', 'coord_center', 'coord_half_range']
    missing = [key for key in required if key not in payload]
    if missing:
        return f'skipped: missing keys: {missing}'

    arch = payload['arch']
    if not isinstance(arch, dict):
        return f'skipped: arch must be dict-like, got {type(arch).__name__}'

    try:
        target_mean = np.asarray(payload['target_mean'], dtype='float32').reshape(-1)
        target_std = np.asarray(payload['target_std'], dtype='float32').reshape(-1)
        coord_center = np.asarray(payload['coord_center'], dtype='float32').reshape(-1)
        coord_half = np.asarray(payload['coord_half_range'], dtype='float32').reshape(-1)
    except Exception as exc:
        return f'skipped: normalization parsing failed: {type(exc).__name__}: {exc}'

    if target_mean.size not in (1, 2) or target_std.size != target_mean.size:
        return f'skipped: unexpected target normalization shapes mean={target_mean.shape} std={target_std.shape}'
    if coord_center.size != 2 or coord_half.size != 2:
        return f'skipped: expected 2D coordinate normalization, got center={coord_center.shape} half={coord_half.shape}'
    if np.any(~np.isfinite(target_mean)) or np.any(~np.isfinite(target_std)):
        return 'skipped: target normalization contains non-finite values'
    if np.any(~np.isfinite(coord_center)) or np.any(~np.isfinite(coord_half)):
        return 'skipped: coordinate normalization contains non-finite values'
    return 'ready'


def discover_checkpoints():
    rows = []
    for regime in DATASETS:
        for family in FAMILIES:
            folder = REPO_ROOT / regime / 'Edge' / family
            scripts = sorted(folder.glob('Training_script*.py')) if folder.exists() else []
            checkpoints = sorted((folder / 'Trained_models').glob('*.pt')) if folder.exists() else []
            if not folder.exists():
                rows.append({
                    'regime': regime,
                    'ablation': 'Edge',
                    'model_family': family,
                    'status': 'missing family directory',
                    'selected': False,
                    'checkpoint_path': None,
                    'training_scripts': [str(x) for x in scripts],
                })
                continue
            if not checkpoints:
                rows.append({
                    'regime': regime,
                    'ablation': 'Edge',
                    'model_family': family,
                    'status': 'missing checkpoint',
                    'selected': False,
                    'checkpoint_path': None,
                    'training_scripts': [str(x) for x in scripts],
                })
                continue
            for path in checkpoints:
                payload = None
                status = 'ready'
                metadata = {}
                try:
                    payload = torch.load(path, map_location='cpu', weights_only=False)
                    status = _required_checkpoint_status(payload)
                    metadata = {
                        'arch': payload.get('arch'),
                        'model_name': payload.get('model_name'),
                        'best_val_loss': payload.get('best_val_loss'),
                        'extra_feat_cols': payload.get('extra_feat_cols', []),
                        'target_names': payload.get('target_names'),
                        'h5_filename': payload.get('h5_filename'),
                        'representation': payload.get('representation'),
                    }
                except Exception as exc:
                    status = f'skipped: {type(exc).__name__}: {exc}'
                rows.append({
                    'regime': regime,
                    'ablation': 'Edge',
                    'model_family': family,
                    'status': status,
                    'selected': False,
                    'checkpoint_path': str(path),
                    'file_size_bytes': path.stat().st_size,
                    'sha256': sha256(path),
                    'training_scripts': [str(x) for x in scripts],
                    **metadata,
                })
    report = pd.DataFrame(rows)
    if report.empty:
        raise RuntimeError('No checkpoint candidates were discovered.')

    for (regime, family), group in report.groupby(['regime', 'model_family'], dropna=False):
        ready = group[group['status'].eq('ready')].copy()
        if ready.empty:
            continue
        if ready['best_val_loss'].notna().any():
            best_idx = ready['best_val_loss'].astype(float).idxmin()
        else:
            best_idx = ready.sort_values('checkpoint_path').index[0]
        report.loc[best_idx, 'selected'] = True
        extra_idx = [idx for idx in ready.index if idx != best_idx]
        if extra_idx:
            report.loc[extra_idx, 'status'] = 'skipped: additional ready checkpoint candidate'
            report.loc[extra_idx, 'selected'] = False

    report = report.sort_values(['regime', 'model_family', 'checkpoint_path'], na_position='last').reset_index(drop=True)
    (RESULTS_DIR / 'checkpoint_integrity.json').write_text(report.to_json(orient='records', indent=2), encoding='utf-8')
    selected = report[report['selected']].copy().reset_index(drop=True)
    eh.save_table(selected[['regime', 'ablation', 'model_family', 'checkpoint_path', 'sha256', 'best_val_loss']], RESULTS_DIR, 'selected_checkpoints')
    return report, selected


checkpoint_report, selected_checkpoints = discover_checkpoints()
display(checkpoint_report[['regime', 'model_family', 'status', 'selected', 'checkpoint_path', 'sha256']])

## HDF5 loading, schema checks, and geometry-level split

The notebook requires local HDF5 assets in `Data_gen/output/`. It verifies the `edge` representation and constructs a deterministic geometry-level holdout with `seed=42` and `fraction=0.20`.

In [ ]:
def load_samples(path):
    samples = []
    with h5py.File(path, 'r') as h5:
        representation = decode(h5.attrs.get('representation', ''))
        if representation != 'edge':
            raise ValueError(f'{path.name}: expected representation edge, got {representation!r}')
        if 'samples' not in h5:
            raise ValueError(f'{path.name}: missing top-level group "samples"')
        for key in sorted(h5['samples'].keys()):
            g = h5['samples'][key]
            def arr(name, default=None):
                return np.asarray(g[name]) if name in g else default
            coords = arr('node_coords_mm')
            stress = arr('stress_max_vm')
            life = arr('life_raw')
            if coords is None or stress is None or life is None:
                raise ValueError(f'{path.name}/{key}: missing required target fields')
            if coords.ndim != 2 or coords.shape[1] != 2:
                raise ValueError(f'{path.name}/{key}: expected node_coords_mm shape [N,2], got {coords.shape}')
            sample_id = decode(g.attrs.get('sample_id', key))
            attrs = {str(k): decode(v) for k, v in g.attrs.items()}
            samples.append({
                'sample_key': key,
                'sample_id': str(sample_id),
                'attrs': attrs,
                'coords': coords.astype('float32'),
                'stress': stress.astype('float32').reshape(-1),
                'loglife': np.log10(np.clip(life.astype('float64').reshape(-1), 1e-30, None)).astype('float32'),
                'zone_id': arr('zone_id', np.full(len(coords), -1)).reshape(-1),
                'subzone_id': arr('subzone_id', np.full(len(coords), np.nan)).reshape(-1),
                'arc_length_mm': arr('arc_length_mm', np.arange(len(coords), dtype='float32')).reshape(-1),
                'node_features': arr('node_features', np.empty((len(coords), 0), dtype='float32')),
            })
    return samples


def split_samples(samples):
    rng = np.random.default_rng(SPLIT_SEED)
    order = rng.permutation(len(samples))
    n_eval = max(1, int(round(len(samples) * EVAL_FRACTION)))
    eval_pos = np.sort(order[:n_eval]).tolist()
    train_pos = np.sort(order[n_eval:]).tolist()
    return train_pos, eval_pos


all_samples, split_records = {}, {}
for regime, path in DATASETS.items():
    if not path.exists():
        raise FileNotFoundError(
            f'Missing required local asset for {regime}: {path}. '
            'Generate or copy the HDF5 before running this notebook.'
        )
    samples = load_samples(path)
    train_pos, eval_pos = split_samples(samples)
    eval_ids = [samples[i]['sample_id'] for i in eval_pos]
    if len(set(eval_ids)) != len(eval_ids):
        raise ValueError(f'{regime}: duplicate sample_id values detected in evaluation split')
    all_samples[regime] = samples
    split_records[regime] = {
        'hdf5_filename': path.name,
        'total_geometry_count': len(samples),
        'evaluation_geometry_count': len(eval_pos),
        'split_seed': SPLIT_SEED,
        'split_fraction': EVAL_FRACTION,
        'training_sample_ids': [samples[i]['sample_id'] for i in train_pos],
        'evaluation_sample_ids': eval_ids,
        'training_positional_indices': train_pos,
        'evaluation_positional_indices': eval_pos,
        'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'notebook_commit_sha': COMMIT,
        'evaluation_label': EVALUATION_LABEL,
        'independence_basis': 'A new geometry holdout is not proved independent of model validation/checkpoint selection.',
    }

with open(RESULTS_DIR / 'evaluation_split_provenance.json', 'w', encoding='utf-8') as stream:
    json.dump(split_records, stream, indent=2)

display(pd.DataFrame([
    {
        'regime': regime,
        'hdf5_filename': data['hdf5_filename'],
        'geometries': data['total_geometry_count'],
        'evaluation_geometries': data['evaluation_geometry_count'],
        'label': data['evaluation_label'],
    }
    for regime, data in split_records.items()
]))

## Model reconstruction, checkpoint validation, and shared-geometry inference

This section reconstructs each selected model using the checkpoint's own architecture, refuses incompatible FP fallbacks, runs inference on the holdout split, and then enforces a **shared geometry ID set across all three families** inside each regime.

In [ ]:
def import_local(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def reconstruct(row):
    folder = Path(row['checkpoint_path']).parent.parent
    family = row['model_family']
    ckpt = torch.load(row['checkpoint_path'], map_location='cpu', weights_only=False)
    arch = dict(ckpt['arch'])
    pn = import_local(folder / 'pn_models.py', f'pn_{row["regime"]}_{family}')
    sys.modules['pn_models'] = pn
    if family == 'PointNetMLPJoint_FP':
        if not hasattr(pn, 'build_fp_model_from_arch'):
            raise RuntimeError('FP checkpoint requires build_fp_model_from_arch; refusing regular PointNet fallback')
        model = pn.build_fp_model_from_arch(arch)
    elif family in ('PointNetMLPJoint', 'PointNetMLPJoint_weighted'):
        model = pn.build_model_from_arch(arch)
    else:
        bench = import_local(folder / 'benchmarks.py', f'bench_{row["regime"]}_{family}')
        if not hasattr(bench, 'ArGEnTDeepONet'):
            raise RuntimeError('No ArGEnTDeepONet found in benchmarks.py')
        defaults = {
            'hidden_dim': 128,
            'num_heads': 4,
            'num_layers': 2,
            'output_dim': 128,
            'out_channels': 1,
            'attention_type': 'self',
            'use_sdf': False,
            'in_ch_geom': 2,
        }
        defaults.update({k: v for k, v in arch.items() if k in defaults})
        if 'out_channels' not in arch and 'bias' in ckpt['model_state']:
            defaults['out_channels'] = int(ckpt['model_state']['bias'].shape[0])
        model = bench.ArGEnTDeepONet(**defaults)
    model.load_state_dict(ckpt['model_state'], strict=True)
    model.eval()
    return model, ckpt


def feature_matrix(sample, ckpt):
    cols = ckpt.get('extra_feat_cols', []) or []
    available = sample['node_features']
    if cols and available.shape[1] < len(cols):
        raise ValueError(f'missing required extra-feature columns: {cols}')
    extra = available[:, :len(cols)] if cols else np.empty((len(sample['coords']), 0), dtype='float32')
    center = np.asarray(ckpt['coord_center'], dtype='float32')
    half = np.asarray(ckpt['coord_half_range'], dtype='float32')
    if center.shape != (2,) or half.shape != (2,):
        raise ValueError(f'expected 2D normalization, got center={center.shape}, half={half.shape}')
    coords = (sample['coords'] - center) / np.maximum(half, 1e-8)
    if extra.shape[1]:
        stats = ckpt.get('extra_feat_stats')
        if stats is None:
            raise ValueError('missing extra-feature normalization statistics')
        if isinstance(stats, dict):
            mean = np.asarray(stats['mean'], dtype='float32')
            std = np.asarray(stats['std'], dtype='float32')
        else:
            mean = np.asarray(stats[0], dtype='float32')
            std = np.asarray(stats[1], dtype='float32')
        extra = (extra - mean) / np.maximum(std, 1e-8)
    return coords, extra


def predict(model, sample, ckpt):
    coords, extra = feature_matrix(sample, ckpt)
    x = torch.from_numpy(coords[None])
    q = x.clone()
    with torch.no_grad():
        try:
            out = model(x, q)
        except TypeError:
            out = model(torch.cat([x, torch.from_numpy(extra[None])], dim=-1), q)
    out = out.detach().cpu().numpy()
    if out.ndim != 3 or out.shape[0] != 1:
        raise ValueError(f'prediction shape unexpected: {out.shape}')
    mean = np.asarray(ckpt['target_mean'])
    std = np.asarray(ckpt['target_std'])
    out = out * std + mean
    if out.shape[2] == 2:
        return out[0, :, 0], out[0, :, 1]
    if out.shape[2] == 1:
        return np.zeros(out.shape[1], dtype='float32'), out[0, :, 0]
    raise ValueError(f'unexpected output channels: {out.shape[2]}')


required_per_regime = pd.MultiIndex.from_product([list(DATASETS.keys()), FAMILIES], names=['regime', 'model_family'])
selected_pairs = pd.MultiIndex.from_frame(selected_checkpoints[['regime', 'model_family']])
missing_pairs = [tuple(x) for x in required_per_regime.difference(selected_pairs).tolist()]
if missing_pairs:
    raise RuntimeError(f'Missing required compatible checkpoints: {missing_pairs}')

node_frames, inference_errors = [], []
for _, row in selected_checkpoints.sort_values(['regime', 'model_family']).iterrows():
    try:
        model, ckpt = reconstruct(row)
        for i in split_records[row['regime']]['evaluation_positional_indices']:
            sample = all_samples[row['regime']][i]
            pred_stress, pred_loglife = predict(model, sample, ckpt)
            if len(pred_stress) != len(sample['coords']) or len(pred_loglife) != len(sample['coords']):
                raise ValueError('prediction length mismatch relative to sample coordinates')
            base = pd.DataFrame({
                'regime': row['regime'],
                'ablation': 'Edge',
                'model_family': row['model_family'],
                'sample_key': sample['sample_key'],
                'sample_id': sample['sample_id'],
                'node_idx': np.arange(len(sample['coords'])),
                'x_mm': sample['coords'][:, 0],
                'r_mm': sample['coords'][:, 1],
                'zone_id': sample['zone_id'],
                'subzone_id': sample['subzone_id'],
                'arc_length_mm': sample['arc_length_mm'],
                'true_stress': sample['stress'],
                'pred_stress': pred_stress,
                'true_loglife': sample['loglife'],
                'pred_loglife': pred_loglife,
                'evaluation_label': EVALUATION_LABEL,
            })
            base['zone_name'] = base['zone_id'].map(eh.ZONE_ID_TO_NAME)
            base['subzone_name'] = base['subzone_id'].map(eh.SUBZONE_ID_TO_NAME)
            node_frames.append(base)
    except Exception as exc:
        inference_errors.append({
            'regime': row['regime'],
            'model_family': row['model_family'],
            'checkpoint_path': row['checkpoint_path'],
            'status': f'skipped during inference: {type(exc).__name__}: {exc}',
        })

if inference_errors:
    with open(RESULTS_DIR / 'inference_errors.json', 'w', encoding='utf-8') as stream:
        json.dump(inference_errors, stream, indent=2)
    raise RuntimeError(f'Inference failed for one or more required checkpoints: {inference_errors}')

node_results = pd.concat(node_frames, ignore_index=True) if node_frames else pd.DataFrame()
if node_results.empty:
    raise RuntimeError('No node-level inference results were produced.')

shared_geometry = {}
for regime in DATASETS:
    fam_to_ids = {}
    for family in FAMILIES:
        ids = sorted(node_results.loc[(node_results['regime'] == regime) & (node_results['model_family'] == family), 'sample_id'].astype(str).unique().tolist())
        if not ids:
            raise RuntimeError(f'{regime}: no inference results for required family {family}')
        fam_to_ids[family] = ids
    shared_ids = set(fam_to_ids[FAMILIES[0]])
    for family in FAMILIES[1:]:
        shared_ids &= set(fam_to_ids[family])
    shared_ids = sorted(shared_ids)
    if not shared_ids:
        raise RuntimeError(f'{regime}: the three families share no common geometry IDs')
    if any(ids != shared_ids for ids in fam_to_ids.values()):
        warnings.warn(f'{regime}: pruning to {len(shared_ids)} shared geometry IDs across all three families')
        node_results = node_results[(node_results['regime'] != regime) | (node_results['sample_id'].isin(shared_ids))].copy()
    shared_geometry[regime] = {
        'families': fam_to_ids,
        'shared_geometry_ids': shared_ids,
        'shared_geometry_count': len(shared_ids),
        'requested_evaluation_geometry_count': len(split_records[regime]['evaluation_sample_ids']),
        'evaluation_label': EVALUATION_LABEL,
    }

node_results = node_results.sort_values(['regime', 'model_family', 'sample_id', 'node_idx']).reset_index(drop=True)
eh.save_table(node_results, RESULTS_DIR, 'node_results')
with open(RESULTS_DIR / 'shared_geometry_ids.json', 'w', encoding='utf-8') as stream:
    json.dump(shared_geometry, stream, indent=2)

display(node_results.groupby(['regime', 'model_family']).sample_id.nunique().reset_index(name='shared_evaluation_geometries'))

## Metrics, paired comparisons, and saved tables

The following cell computes pooled metrics, low-life bins, principal-subzone metrics, geometry-level summaries, and the required paired comparisons where **positive means FP is better**.

In [ ]:
pooled_metrics = eh.pooled_metrics_from_nodes(node_results)
low_life_bins = eh.loglife_bin_metrics(node_results)
subzone_metrics = eh.zone_metrics_from_nodes(node_results)
geometry_metrics = eh.geometry_level_metrics(node_results)
geometry_summary = eh.geometry_metrics_summary(geometry_metrics)

for frame in [pooled_metrics, low_life_bins, subzone_metrics, geometry_metrics, geometry_summary]:
    if not frame.empty:
        frame.insert(0, 'evaluation_label', EVALUATION_LABEL)

for name, frame in [
    ('pooled_metrics', pooled_metrics),
    ('low_life_bins', low_life_bins),
    ('subzone_metrics', subzone_metrics),
    ('geometry_level_metrics', geometry_metrics),
    ('geometry_level_summary', geometry_summary),
]:
    eh.save_table(frame, RESULTS_DIR, name)


def paired_diff(geom_df, left_family, right_family, out_name):
    rows = []
    for regime, sub in geom_df.groupby('regime'):
        left = sub[sub['model_family'] == left_family].set_index('sample_id')
        right = sub[sub['model_family'] == right_family].set_index('sample_id')
        common = sorted(set(left.index).intersection(right.index))
        for sample_id in common:
            a = left.loc[sample_id]
            b = right.loc[sample_id]
            rows.append({
                'evaluation_label': EVALUATION_LABEL,
                'regime': regime,
                'ablation': 'Edge',
                'sample_id': sample_id,
                'left_family': left_family,
                'right_family': right_family,
                'left_abs_min_loglife_error': float(abs(a['min_loglife_error_decades'])),
                'right_abs_min_loglife_error': float(abs(b['min_loglife_error_decades'])),
                'left_minus_right': float(abs(a['min_loglife_error_decades']) - abs(b['min_loglife_error_decades'])),
                'positive_means': 'FP better' if right_family == 'PointNetMLPJoint_FP' else 'right family better',
                'output_name': out_name,
            })
    return pd.DataFrame(rows)


paired_argent_minus_fp = paired_diff(geometry_metrics, 'ArGEnT_self_att_noSDF', 'PointNetMLPJoint_FP', 'paired_argent_minus_fp')
paired_pointnet_minus_fp = paired_diff(geometry_metrics, 'PointNetMLPJoint', 'PointNetMLPJoint_FP', 'paired_pointnet_minus_fp')

for name, frame in [
    ('paired_argent_minus_fp', paired_argent_minus_fp),
    ('paired_pointnet_minus_fp', paired_pointnet_minus_fp),
]:
    eh.save_table(frame, RESULTS_DIR, name)

paired_summary = []
for label, frame in [('ArGEnT - FP', paired_argent_minus_fp), ('PointNetMLPJoint - FP', paired_pointnet_minus_fp)]:
    for regime, sub in frame.groupby('regime'):
        paired_summary.append({
            'evaluation_label': EVALUATION_LABEL,
            'regime': regime,
            'comparison': label,
            'n_geometries': int(len(sub)),
            'median_left_minus_right': float(sub['left_minus_right'].median()),
            'fraction_positive_fp_better': float((sub['left_minus_right'] > 0).mean()),
        })
paired_summary = pd.DataFrame(paired_summary)
eh.save_table(paired_summary, RESULTS_DIR, 'paired_summary')

run_metadata = {
    'commit_sha': COMMIT,
    'software_versions': VERSIONS,
    'evaluation_label': EVALUATION_LABEL,
    'results_dir': str(RESULTS_DIR),
    'families': FAMILIES,
    'datasets': {k: str(v) for k, v in DATASETS.items()},
}
eh.save_json(run_metadata, RESULTS_DIR, 'run_metadata')

display(pooled_metrics)
display(paired_summary)

## Representative geometries

Representative geometries are selected per regime from the shared evaluated IDs:
- **median** composite behaviour,
- **lowest true minimum life**,
- **greatest ArGEnT–FP disagreement** in absolute minimum-life error.

In [ ]:
representatives = {}
representative_rows = []
for regime in DATASETS:
    picks = eh.select_representative_geometries(
        geometry_metrics,
        regime,
        'Edge',
        disagreement_family='PointNetMLPJoint_FP',
    )
    representatives[regime] = picks
    regime_geom = geometry_metrics[geometry_metrics['regime'] == regime]
    for selection, sample_id in picks.items():
        if sample_id is None:
            continue
        ref = regime_geom[regime_geom['sample_id'].astype(str) == str(sample_id)].iloc[0]
        representative_rows.append({
            'evaluation_label': EVALUATION_LABEL,
            'regime': regime,
            'selection': selection,
            'sample_id': sample_id,
            'true_min_loglife': float(ref['true_min_loglife']),
            'true_min_life_cycles': float(eh.loglife_to_raw_life(ref['true_min_loglife'])),
            'true_max_stress_mpa': float(ref['true_max_stress']),
            'governing_zone': ref['true_crit_zone'],
            'governing_subzone': ref['true_crit_subzone'],
        })

representative_table = pd.DataFrame(representative_rows)
eh.save_table(representative_table, RESULTS_DIR, 'representative_geometries')
eh.save_json({'evaluation_label': EVALUATION_LABEL, 'representatives': representatives}, RESULTS_DIR, 'representative_geometry_ids')

display(representative_table)

## Figures

This cell writes the required bar charts, geometry scatter plots, field-comparison maps, and arc-length error plots into `Comparison/results/01_fp_vs_argent/figures/`.

In [ ]:
for regime in DATASETS:
    regime_fig_dir = FIGURES_DIR / regime.lower()
    regime_fig_dir.mkdir(parents=True, exist_ok=True)

    zone_df = subzone_metrics[subzone_metrics['regime'] == regime].copy()
    bin_df = low_life_bins[low_life_bins['regime'] == regime].copy()
    geom_df = geometry_metrics[geometry_metrics['regime'] == regime].copy()

    eh.plot_zone_bar(zone_df, f'{regime} Edge subzone LogLife MAE ({EVALUATION_LABEL})', out_dir=regime_fig_dir, filename='subzone_bar')
    eh.plot_bin_bar(bin_df, f'{regime} Edge low-life bins ({EVALUATION_LABEL})', out_dir=regime_fig_dir, filename='low_life_bins')
    eh.plot_geometry_scatter(geom_df, out_dir=regime_fig_dir, filename_prefix='geometry')

    for selection, sample_id in representatives.get(regime, {}).items():
        if sample_id is None:
            continue
        by_model = {
            family: node_results[
                (node_results['regime'] == regime)
                & (node_results['model_family'] == family)
                & (node_results['sample_id'].astype(str) == str(sample_id))
            ].copy()
            for family in FAMILIES
        }
        if any(frame.empty for frame in by_model.values()):
            warnings.warn(f'Skipping representative {regime}/{selection}/{sample_id}: incomplete model coverage')
            continue
        loglife_limit = eh.robust_signed_limit([
            frame['pred_loglife'].to_numpy() - frame['true_loglife'].to_numpy()
            for frame in by_model.values()
        ], pct=99.0, floor=0.05)
        stress_limit = eh.robust_signed_limit([
            frame['pred_stress'].to_numpy() - frame['true_stress'].to_numpy()
            for frame in by_model.values()
        ], pct=99.0, floor=1.0)

        stem = f'{selection}_{sample_id}'
        eh.plot_field_comparison(
            by_model,
            'true_loglife',
            'pred_loglife',
            'decades',
            f'{regime} {selection} sample {sample_id} LogLife ({EVALUATION_LABEL})',
            fixed_error_limit=loglife_limit,
            mark_extrema='min',
            out_dir=regime_fig_dir,
            filename=f'{stem}_loglife_map',
        )
        eh.plot_field_comparison(
            by_model,
            'true_stress',
            'pred_stress',
            'MPa',
            f'{regime} {selection} sample {sample_id} Stress ({EVALUATION_LABEL})',
            fixed_error_limit=stress_limit,
            mark_extrema='max',
            out_dir=regime_fig_dir,
            filename=f'{stem}_stress_map',
        )
        eh.plot_arc_length_error(
            by_model,
            f'{regime} {selection} sample {sample_id} arc-length errors ({EVALUATION_LABEL})',
            out_dir=regime_fig_dir,
            filename=f'{stem}_arc_length_error',
        )
        plt.close('all')

## Artifact listing

A final machine-readable artifact listing is written to the results directory for provenance tracking.

In [ ]:
def build_artifact_listing(root):
    rows = []
    for path in sorted(root.rglob('*')):
        if path.is_file():
            rows.append({
                'relative_path': str(path.relative_to(root)),
                'size_bytes': int(path.stat().st_size),
            })
    return rows


artifact_listing = build_artifact_listing(RESULTS_DIR)
eh.save_json(artifact_listing, RESULTS_DIR, 'artifact_listing')
display(pd.DataFrame(artifact_listing[:25]))

## Conclusion

For the currently tracked Uniform/Edge and Zonal/Edge checkpoints, **`PointNetMLPJoint_FP` is the strongest overall model in this validation-split evaluation**. It ranks first on pooled stress/log-life error, low-life-bin error, lower-transition/subzone error, per-geometry minimum-life error, and maximum-stress recovery; `PointNetMLPJoint` is usually second, while `ArGEnT_self_att_noSDF` trails on the fatigue-critical local metrics. The metrics are not redundant: pooled R² stays very high for all models, while raw-life `Max_PE (%)` can disagree because a few very small true-life values dominate percentage error. Taken together, the agreement across pooled, low-life, lower-transition, paired, and representative-geometry diagnostics supports FP as useful for **local fatigue-critical prediction**, not only for global field fit. Re-check the generated tables if checkpoints or datasets change.